# Transfer Learning (ResNet18) on CIFAR-10

This notebook explains how we used a pretrained CNN (ResNet18) and adapted it to CIFAR-10. The goal is to compare transfer learning against our from-scratch CNN results.

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from SRC.train_transfer_cnn import build_model

TRANSFER_PATH = PROJECT_ROOT / "Data" / "transfer_learning_outputs" / "run_20260401_232915" / "summary.json"
TAKE4_PATH = PROJECT_ROOT / "Data" / "improved_cnn_outputs" / "cnn_take_4_20260326_234323" / "summary.json"
BASELINE_PATH = PROJECT_ROOT / "Data" / "cnn_baseline_outputs" / "run_20260325_130955" / "summary.json"

with TRANSFER_PATH.open("r", encoding="utf-8") as handle:
    transfer = json.load(handle)
with TAKE4_PATH.open("r", encoding="utf-8") as handle:
    take4 = json.load(handle)
with BASELINE_PATH.open("r", encoding="utf-8") as handle:
    baseline = json.load(handle)

transfer["test_accuracy"], take4["test_accuracy"]

## 1. What Transfer Learning Means Here

We start from a CNN pretrained on ImageNet and adapt it to CIFAR-10 by replacing the final classifier layer. That lets us reuse strong visual features (edges, textures, shapes) learned from a huge dataset.

## 2. Training Setup Used

For this run we used a **ResNet18** with:

- Image resize to `224x224` to match ImageNet features
- ImageNet normalization
- Feature extraction (freeze backbone, train only the head)
- Light augmentation (flip + color jitter)

This is the simplest, safest transfer learning baseline for CIFAR-10.

In [ ]:
pd.Series(
    {
        "model": transfer["model"],
        "resize": transfer["resize"],
        "epochs": transfer["epochs"],
        "batch_size": transfer["batch_size"],
        "learning_rate": transfer["learning_rate"],
        "weight_decay": transfer["weight_decay"],
        "freeze_backbone": transfer["freeze_backbone"],
        "unfreeze_epoch": transfer["unfreeze_epoch"],
        "augment": transfer["augment"],
        "best_val_accuracy": transfer["best_val_accuracy"],
        "test_accuracy": transfer["test_accuracy"],
    },
    name="transfer_learning",
)

## 3. Compare Against From-Scratch CNNs

We compare transfer learning to both the baseline CNN and CNN Take 4.

In [ ]:
comparison = pd.DataFrame(
    [
        {"run": "Baseline CNN", "test_accuracy": baseline["test_accuracy"], "best_val_accuracy": baseline["best_val_accuracy"]},
        {"run": "CNN Take 4", "test_accuracy": take4["test_accuracy"], "best_val_accuracy": take4["best_val_accuracy"]},
        {"run": "Transfer ResNet18", "test_accuracy": transfer["test_accuracy"], "best_val_accuracy": transfer["best_val_accuracy"]},
    ]
)

display(comparison)

plt.figure(figsize=(8, 4))
plt.bar(comparison["run"], comparison["test_accuracy"], color=["#6b8bd6", "#d45a5a", "#5ca46a"])
plt.axhline(0.80, color="black", linestyle="--", linewidth=1, label="80% target")
plt.ylabel("Test Accuracy")
plt.ylim(0.5, 0.9)
plt.title("Transfer Learning vs Scratch CNNs")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Interpretation

- The transfer learning run already performs much better than the original baseline CNN.
- CNN Take 4 still leads in this project because it uses a strong augmentation and training recipe tuned for CIFAR-10.
- This transfer run only completed a few epochs so far; longer fine-tuning could potentially catch or exceed Take 4.

This gives us a clean, report-friendly comparison: scratch CNN vs transfer learning.

## 5. Key Files

- Transfer learning trainer: `SRC/train_transfer_cnn.py`
- Transfer learning run: `Data/transfer_learning_outputs/run_20260401_232915/summary.json`
- CNN Take 4 run: `Data/improved_cnn_outputs/cnn_take_4_20260326_234323/summary.json`